# ACS in-hospital mortality prediction

This notebook evaluates a 13-feature Random Forest in 1,524 patients with acute coronary syndrome. The primary validation uses five stratified folds repeated with ten fixed seeds. All preprocessing is estimated within each training fold.

The patient-level mean of the ten out-of-fold predictions is used only for threshold selection and the paired GRACE 2.0 comparison. Seed-level AUC variation is retained for uncertainty reporting.

## TRIPOD+AI reporting checklist

| Item | Implementation |
|---|---|
| Population and outcome | ACS, Killip I to III, in-hospital mortality |
| Sample size | 1,524 patients, 115 deaths |
| Predictors | 13 admission variables specified before analysis |
| Missing data | Training-fold median imputation |
| Internal validation | Five folds repeated over ten fixed seeds |
| Performance | AUC, 95% CI, range, AUPRC, Brier score, threshold metrics |
| Comparator | Patient-level GRACE 2.0 score from eight source variables |
| Model interpretation | Mean Gini importance over all 50 validation models |
| Reproducibility | Versions, seeds, filters, and schema reported below |

## 1. Execute the canonical analysis

The standalone pipeline is the single source of numerical results. Running it here prevents notebook outputs from drifting away from the script, figures, or thesis tables.

In [ ]:
import subprocess, sys
run = subprocess.run([sys.executable, "thesis_main.py"], check=True, text=True, capture_output=True)
print(run.stdout)

The execution log reports one AUC for each prespecified seed. Its final lines report the seed-level summary, exact thresholds, paired GRACE comparison, and mutually exclusive triage counts.

## 2. Load the synchronized results

The JSON artifact contains full-precision values. Display rounding is applied only when results are presented.

In [ ]:
import json
from pathlib import Path
r = json.loads(Path("validation_results.json").read_text())
d, m, t, g = r["dataset"], r["metrics"], r["thresholds"], r["grace"]
print(f'N={d["n"]:,}; deaths={d["deaths"]} ({d["prevalence"]:.1%})')
print(f'Seed AUC={m["auc_mean"]:.4f} ± {m["auc_sd"]:.4f}; 95% CI {m["auc_ci95"][0]:.4f} to {m["auc_ci95"][1]:.4f}')
print(f'Range={m["auc_min"]:.4f} to {m["auc_max"]:.4f}; mean-OOF AUC={m["ensemble_auc"]:.4f}')
print(f'Brier={m["brier_mean"]:.4f}; AUPRC={m["auprc_mean"]:.4f}')

The mean seed AUC quantifies performance across repeated partitions. The mean-OOF AUC is slightly higher because averaging ten patient-level predictions reduces prediction noise. The Brier score and AUPRC replace the stale values previously shown in this notebook.

## 3. Baseline characteristics

Continuous variables use Welch's t-test. Binary rows use a 2 by 2 chi-square or Fisher exact test. The separate Killip helper uses the full 2 by 3 table, so Killip classes I, II, and III are retained in the multicategory test.

In [ ]:
import pandas as pd
baseline = pd.DataFrame(r["baseline"])
baseline["p"] = baseline["p"].map(lambda x: f"{x:.4f}")
display(baseline)
print(f'Killip I to III multicategory p={r["killip_multicategory_p"]:.6g}')

Deaths were associated with an older and physiologically higher-risk profile across several admission variables. The binary Killip III row remains suitable for the clinical baseline table, while the full multicategory test evaluates the complete Killip distribution.

## 4. Threshold selection and clinical triage

The safety threshold is the largest observed cutoff that permits no more than two false negatives among 115 deaths. The Youden threshold maximizes sensitivity plus specificity minus one. Both thresholds are derived from the patient-level mean OOF vector.

In [ ]:
s, y = t["safety_metrics"], t["youden_metrics"]
print(f'Safety threshold={t["safety"]:.6f}: sensitivity={s["sensitivity"]:.1%}, specificity={s["specificity"]:.1%}, FN={s["fn"]}, FP={s["fp"]}')
print(f'Youden threshold={t["youden"]:.6f}: sensitivity={y["sensitivity"]:.1%}, specificity={y["specificity"]:.1%}, FN={y["fn"]}, FP={y["fp"]}')
triage = pd.DataFrame(t["triage"]).T
triage["mortality"] = triage["rate"].map(lambda x: f"{x:.1%}")
display(triage[["n", "death", "mortality"]])

The safety rule assigns 371 patients to Ward and misses two deaths. The Youden cutoff separates 336 ICU patients with 24.4% observed mortality. These are validation-derived decision rules and require external assessment before clinical deployment.

## 5. GRACE 2.0 comparison

GRACE points are calculated from age, heart rate, systolic pressure, creatinine, Killip class, arrest at admission, elevated troponin, and ST deviation. Arrest is mapped to `rosc_in_igd`, elevated enzymes to troponin above 0.04 ng/mL, and ST deviation to `ekg_ste`. Six missing creatinine values use the cohort median for this fixed-score comparator.

In [ ]:
q = g["threshold_20pct"]
comparison = pd.DataFrame({"Random Forest": q["rf"], "GRACE 2.0": q["grace"]})
display(comparison.loc[["sensitivity", "specificity", "ppv", "npv", "tn", "fp", "fn", "tp"]])
print(f'RF mean-OOF AUC={g["rf_auc"]:.4f}; GRACE AUC={g["auc"]:.4f}; delta={g["auc_delta"]:.4f}')
print(f'Bootstrap delta 95% CI={g["auc_delta_ci95"][0]:.4f} to {g["auc_delta_ci95"][1]:.4f}; p={g["auc_p_bootstrap"]:.4g}')
print(f'GRACE 20% risk begins at score {q["grace_score_min"]}; McNemar p={q["mcnemar_p"]:.4g}')

The Random Forest mean-OOF vector discriminates mortality better than the reconstructed GRACE score in this cohort. At a 20% predicted-risk cutoff, the paired McNemar test evaluates whether patient-level classification errors differ between methods.

## 6. Cross-validated feature importance

Importance is aggregated over the same 50 fitted fold models used for validation. No full-cohort refit contributes to this table.

In [ ]:
importance = pd.DataFrame(r["feature_importance"]).T.reset_index(names="feature")
display(importance.style.format({"mean": "{:.4f}", "sd": "{:.4f}"}))

Renal function and age have the largest mean impurity reductions. Gini importance ranks model usage rather than causal effect, and correlated predictors can share importance.

## 7. Secondary outcomes: de novo cardiogenic shock and composite

In [ ]:
with open('three_outcomes_results.json') as f:
    three = json.load(f)

outcome_rows = [
    ('Mortality', three['mortality'], 115, 7.5),
    ('De novo shock', three['shock'], 171, 11.2),
    ('Composite', three['composite'], 197, 12.9),
]
print('Cohort: N=1,524; deaths=115 (7.5%); de novo shock=171 (11.2%); composite=197 (12.9%)')
secondary = pd.DataFrame([
    {
        'Outcome': label,
        'Events': events,
        'Prevalence': f'{prevalence:.1f}%',
        'AUC': f"{values['mean']:.3f}",
        'SD': f"{values['std']:.3f}",
        '95% CI': f"{values['ci_lo']:.3f}-{values['ci_hi']:.3f}",
    }
    for label, values, events, prevalence in outcome_rows
])
display(secondary)
print('AUPRC: mortality=0.301; de novo shock=0.500; composite=0.635')
print('Top-3 feature importance:')
print(f"  Mortality: eGFR ({three['mortality']['feature_importance']['egfr_igd']:.3f}), ureum ({three['mortality']['feature_importance']['ureum_igd']:.3f}), LVOT VTI ({three['mortality']['feature_importance']['lvot_vti_igd']:.3f})")
print(f"  De novo shock: LVOT VTI ({three['shock']['feature_importance']['lvot_vti_igd']:.3f}), SBP ({three['shock']['feature_importance']['sbp']:.3f}), APTT ({three['shock']['feature_importance']['aptt_value']:.3f})")
print(f"  Composite: LVOT VTI ({three['composite']['feature_importance']['lvot_vti_igd']:.3f}), SBP ({three['composite']['feature_importance']['sbp']:.3f}), eGFR ({three['composite']['feature_importance']['egfr_igd']:.3f})")

Mortality had the strongest discrimination. For the secondary outcomes, AUPRC was 0.500 for de novo shock and 0.635 for the composite, compared with prevalence baselines of 0.112 and 0.129. Feature-importance rankings were outcome-specific and describe model usage rather than causal effects.

## 10. Reproducibility record

The following cell prints package versions, exact split seeds, cohort filter, and feature schema. The parquet file is read-only throughout the workflow.

In [ ]:
print('Versions:', r["versions"])
print('Seeds:', r["seeds"])
print('Cohort filter:', r["dataset"]["filter"])
print('Feature schema:', r["features"])
print('Threshold method:', r["thresholds"]["method"])

The recorded environment and fixed seeds support exact reruns in the current repository. The schema lists all model inputs and distinguishes the modeling features from the separate GRACE variables.

## 11. Final synchronized metrics

This compact table is the reference for the thesis narrative and validation script.

In [ ]:
summary = pd.DataFrame([
    ["Patients", d["n"]], ["Deaths", d["deaths"]],
    ["Mean seed AUC", f'{m["auc_mean"]:.4f}'], ["Mean-OOF AUC", f'{m["ensemble_auc"]:.4f}'],
    ["Brier", f'{m["brier_mean"]:.4f}'], ["AUPRC", f'{m["auprc_mean"]:.4f}'],
    ["Safety threshold", f'{t["safety"]:.6f}'], ["Youden threshold", f'{t["youden"]:.6f}'],
    ["GRACE AUC", f'{g["auc"]:.4f}'], ["McNemar p", f'{g["threshold_20pct"]["mcnemar_p"]:.4g}']])
display(summary.rename(columns={0: "Metric", 1: "Value"}))

All values in this summary are generated from the corrected pipeline. The thesis headline can state an OOF AUC of approximately 0.818, while the methods and results must also report the seed-level mean, uncertainty interval, and range shown above.